# Bagging intuition

**Draw rows with replacement → fit one tree per sample → predict the same email → combine the predictions.**

We use the same 12 labeled emails and the same three saved samples as the [Bagging lesson](https://sharanharsoor.github.io/ml-learning-lab/bagging/#forest). All three trees predict **E574**, which stays outside their training data.

The two input columns are percentages of **characters**. For example, one dollar sign in 1,000 characters is 0.1%. The label is 0 for Legitimate and 1 for Spam.

The setup cell loads the two measurements, the labels, and the saved training samples. Run it once, then start explaining at **Draw rows with replacement**.

[UCI Spambase](https://doi.org/10.24432/C53G6X), Mark Hopkins, Erik Reeber, George Forman, and Jaap Suermondt (1999). [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier

training = pd.DataFrame([
    ('E3590', 0, 0, 0),
    ('E3383', 0.024, 0, 0),
    ('E4256', 0, 0.278, 0),
    ('E2321', 0, 0.089, 0),
    ('E3066', 0, 0.244, 0),
    ('E3776', 0.023, 0, 0),
    ('E779', 0.145, 0.393, 1),
    ('E1375', 0.231, 0.925, 1),
    ('E1748', 0, 0.518, 1),
    ('E1188', 0.312, 0.625, 1),
    ('E319', 0.107, 0.268, 1),
    ('E1153', 0.073, 0.25, 1),
], columns=['email_id', 'dollar_pct', 'exclamation_pct', 'label']).set_index('email_id')

features = ['dollar_pct', 'exclamation_pct']
X = training[features]
y = training['label']
names = {0: 'Legitimate', 1: 'Spam'}
email = pd.DataFrame([(0.034, 0.375)], columns=features, index=['E574'])

# Saved draw order for the three trees in the lesson.
demo_draws = [
    ['E3776', 'E3776', 'E1748', 'E4256', 'E1188', 'E1153', 'E2321', 'E4256', 'E1153', 'E3066', 'E3776', 'E1375'],
    ['E3590', 'E319', 'E3776', 'E779', 'E3590', 'E3776', 'E4256', 'E779', 'E2321', 'E1748', 'E1748', 'E3590'],
    ['E1748', 'E319', 'E3383', 'E2321', 'E1153', 'E1748', 'E3776', 'E1748', 'E1375', 'E1375', 'E3383', 'E4256'],
]

print(f'{len(X)} labeled training emails')
display(email)

## Draw rows with replacement

Each ID is a ticket. After each draw, the ticket goes back into the pool. This gives it another chance to appear. The sample still has 12 draws, and some IDs can appear more than once.

Run this cell once with `replace=True`, then try `replace=False` and compare the number of distinct IDs. This experiment uses a fresh draw. The next section replays Tree 1’s saved draw order so it matches the lesson.

In [ ]:
rng = np.random.default_rng(11)
fresh_draw = rng.choice(X.index, size=len(X), replace=True)

print('Draw order:', ', '.join(fresh_draw))
print(f'{len(fresh_draw)} draws, {len(set(fresh_draw))} distinct emails')

## Fit one tree and predict E574

- `demo_draws[0]` is the same tray shown for Tree 1. It starts with the same email twice.
- `X.loc[draw]` and `y.loc[draw]` pull the features and the label in matching order, including repeats.
- `fit` learns the questions from those rows. A maximum depth of 2 allows up to two questions on a path. `predict` applies the learned questions to E574.

Follow the same path in Tree 1’s diagram. E574 has 0.034% dollar signs and 0.375% exclamation marks. Both values are at or below Tree 1’s cutoffs of 0.048% and 0.398%, so the tree follows Yes twice and predicts Legitimate.

In [ ]:
draw = demo_draws[0]
tree = DecisionTreeClassifier(criterion='gini', max_depth=2, random_state=0)
tree.fit(X.loc[draw], y.loc[draw])
prediction = tree.predict(email)[0]

print('First two draws:', ', '.join(draw[:2]))
print(f'{len(draw)} draws, {len(set(draw))} distinct emails')
print(f'Tree 1 predicts {names[prediction]} for E574.')

## Fit one tree per sample

Repeat the same fit on each saved tray. Every tree gets the same E574 inputs when it predicts. A tree’s **Spam probability** is the fraction of Spam training draws in the leaf E574 reaches. The values returned by `predict_proba` follow the order in `classes_`.

In [ ]:
results = []

for number, draw in enumerate(demo_draws, start=1):
    model = DecisionTreeClassifier(criterion='gini', max_depth=2, random_state=0)
    model.fit(X.loc[draw], y.loc[draw])
    vote = names[model.predict(email)[0]]
    probabilities = dict(zip(model.classes_, model.predict_proba(email)[0]))
    results.append({
        'Tree': number,
        'Vote': vote,
        'Spam probability': probabilities.get(1, 0.0),
    })

results = pd.DataFrame(results).set_index('Tree')
display(results.round(3))

## Combine the trees’ predictions

**Hard vote** counts each tree’s predicted class once. The first two trees tie, and the third gives Spam the majority. **Average probability** uses the Spam fractions from the three leaves, `0/7`, `4/4`, and `2/3`. Their mean is `5/9`, about 55.6%. That’s above 50%, so both rules predict Spam here.

Read the ground truth for E574 after seeing both results. This example shows how the methods combine predictions. The full notebook checks accuracy across many held-out emails.

In [ ]:
votes = results['Vote'].value_counts()
mean_spam = results['Spam probability'].mean()

print('Hard votes:', votes.to_dict())
print(f'Mean Spam probability: {mean_spam:.1%}')
print('Ground truth for E574:', names[1])

## Where Random Forest adds a feature draw

These three fitted trees demonstrate **bagging**. Each trains on its own saved bootstrap sample and can consider both features at every split. A **Random Forest** also samples a set of candidate features afresh at each split.

[Open the full Random Forest implementation in Colab](https://colab.research.google.com/github/sharanharsoor/ml-learning-lab/blob/main/labs/bagging/notebooks/random-forest-implementation.ipynb)